# Network Protocols and Quality

A usable connection needs more than an address. This notebook follows a hostname to a service, distinguishes transport protocols, and describes the quality measures that affect live Duckiedrone data.

## From a name to a service

A connection to a Duckiedrone has several distinct steps, shown in [Figure 1](#figure-1).

<figure id="figure-1" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    hostname
      |
      v
    address lookup
      |
      v
    route
      |
      v
    reachable Duckiedrone
      |
      v
    protocol and port
      |
      v
    service
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">Figure 1: Steps from a hostname to a network service.</figcaption>
</figure>

A __resolver__ translates `DUCKIEDRONE_NAME.local` into an IP address. The operating system chooses a route, directly on the local subnet or through a default gateway. The Duckiedrone must be reachable across that route, and the expected service must listen for the expected protocol and port.

An __endpoint__ identifies a service with a transport protocol, hostname or IP address, and port. For example, `DUCKIEDRONE_NAME.local:22` commonly identifies an SSH service using Transmission Control Protocol (TCP) port `22`. An IP address alone does not identify which service, if any, is listening.

A __client-server model__ is a common pattern for networked software: a client requests a service, and a server listens for and responds to those requests. For example, an SSH client connects to an SSH server at a known endpoint.

A __distributed application__ is made of independent processes that communicate through defined interfaces, often across a network. Its parts can run on one computer or several computers.

A __software interface__ is an agreed way for software components to interact. An __Application Programming Interface (API)__ is an interface used by program code, while an __application protocol__ defines messages sent across a network. A __data contract__ specifies the fields and meanings both sides expect; TCP and User Datagram Protocol (UDP) transport data but do not define that format.

## Ports and transport protocols

A __port__ identifies a service on a network interface. One Duckiedrone can use one IP address for several services because each listens on a different port. Port numbers have meaning together with a transport protocol: TCP port `22` and UDP port `22` are different endpoints.

TCP provides an ordered, reliable byte stream between endpoints. SSH and ordinary web dashboards use TCP. UDP sends independent datagrams without the same delivery or ordering guarantees; mDNS uses UDP to ask nearby devices about `.local` names. `ping` uses Internet Control Message Protocol (ICMP), not TCP or UDP, and ICMP has no service port. A successful ping does not prove TCP is available, and a successful TCP connection does not prove ICMP is available.

## Network quality

Reachability asks whether some packets can travel between computers. __Network quality__ describes how quickly and consistently they travel:

- __Latency__ is travel time. Lower latency usually makes remote control and interactive feedback feel more immediate. `ping` reports a round-trip time, one rough latency measurement.

- __Bandwidth__ is the amount of data a connection can carry in a given time. It affects camera, dashboard, and other high-rate data, but high bandwidth does not automatically mean low latency.

- __Jitter__ is variation in latency. Changing delay can make live data arrive unevenly even when average latency is acceptable.

- __Packet loss__ means some packets do not arrive. TCP can request missing data again, adding delay; UDP has no equivalent recovery, so loss can appear as missing or late live data.

A Duckiedrone can be reachable while a camera stream or dashboard performs poorly. An SSH terminal uses little data and can remain usable when higher-rate data is delayed, uneven, or interrupted. Wi-Fi quality changes with distance, obstacles, interference, and the number of devices sharing a channel; a wired connection is often more stable but is not a guarantee. Do not create artificial load or run performance tests unless you own the network or its owner or administrator has explicitly authorized them. Record observed behavior, time, location, and an authorized targeted check before asking the device owner, network administrator, or other appropriate support contact for help.

## How a packet crosses a network

Networking works in layers. [Figure 2](#figure-2) shows how an SSH connection is encapsulated for a local link.

<figure id="figure-2" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    SSH data
      |
      v
    TCP: source and destination ports
      |
      v
    IP: source and destination IP addresses
      |
      v
    Ethernet or Wi-Fi: MAC addresses for the current local link
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">Figure 2: Network layers carrying an SSH connection.</figcaption>
</figure>

TCP identifies the services, IP identifies the intended source and destination across networks, and Ethernet or Wi-Fi identifies the next device on the current local link. For an IPv4 destination on the same subnet, the base station creates a frame addressed directly to the Duckiedrone's MAC address. For an off-subnet destination, it addresses the first frame to the default gateway's MAC address while preserving the final destination IP in the packet. Each router builds a new frame for the next link, so MAC addresses can change at every router while the final IP address and port continue to identify the intended service.

Before creating an IPv4 frame, a computer can use the Address Resolution Protocol (ARP) to learn the MAC address of its next hop. This local step is another reason a known Duckiedrone IP address does not guarantee a usable connection.

## Further reading

The Internet Engineering Task Force (IETF) [packet delay variation](https://www.rfc-editor.org/rfc/rfc3393) specification provides detail on jitter.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
